<a href="https://colab.research.google.com/github/kartikanand73/AnthropicCertification/blob/main/evaluator_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Evaluator-Optimizer Workflow
In this workflow, one LLM call generates a response while another provides evaluation and feedback in a loop.

### When to use this workflow
This workflow is particularly effective when we have:

- Clear evaluation criteria
- Value from iterative refinement

The two signs of good fit are:

- LLM responses can be demonstrably improved when feedback is provided
- The LLM can provide meaningful feedback itself

In [ ]:
!wget -q https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/patterns/agents/util.py
!pip install anthropic -q

import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 11.0 MB/s eta 0:00:00


In [ ]:
from util import extract_xml, llm_call


def generate(prompt: str, task: str, context: str = "") -> tuple[str, str]:
    """Generate and improve a solution based on feedback."""
    full_prompt = f"{prompt}\n{context}\nTask: {task}" if context else f"{prompt}\nTask: {task}"
    response = llm_call(full_prompt)
    thoughts = extract_xml(response, "thoughts")
    result = extract_xml(response, "response")

    print("\n=== GENERATION START ===")
    print(f"Thoughts:\n{thoughts}\n")
    print(f"Generated:\n{result}")
    print("=== GENERATION END ===\n")

    return thoughts, result


def evaluate(prompt: str, content: str, task: str) -> tuple[str, str]:
    """Evaluate if a solution meets requirements."""
    full_prompt = f"{prompt}\nOriginal task: {task}\nContent to evaluate: {content}"
    response = llm_call(full_prompt)
    evaluation = extract_xml(response, "evaluation")
    feedback = extract_xml(response, "feedback")

    print("=== EVALUATION START ===")
    print(f"Status: {evaluation}")
    print(f"Feedback: {feedback}")
    print("=== EVALUATION END ===\n")

    return evaluation, feedback


def loop(task: str, evaluator_prompt: str, generator_prompt: str) -> tuple[str, list[dict]]:
    """Keep generating and evaluating until requirements are met."""
    memory = []
    chain_of_thought = []

    thoughts, result = generate(generator_prompt, task)
    memory.append(result)
    chain_of_thought.append({"thoughts": thoughts, "result": result})

    while True:
        evaluation, feedback = evaluate(evaluator_prompt, result, task)
        if evaluation == "PASS":
            return result, chain_of_thought

        context = "\n".join(
            ["Previous attempts:", *[f"- {m}" for m in memory], f"\nFeedback: {feedback}"]
        )

        thoughts, result = generate(generator_prompt, task, context)
        memory.append(result)
        chain_of_thought.append({"thoughts": thoughts, "result": result})

### Example Use Case: Iterative coding loop



In [ ]:
evaluator_prompt = """
Evaluate this following code implementation for:
1. code correctness
2. time complexity
3. style and best practices

You should be evaluating only and not attemping to solve the task.
Only output "PASS" if all criteria are met and you have no further suggestions for improvements.
Output your evaluation concisely in the following format.

<evaluation>PASS, NEEDS_IMPROVEMENT, or FAIL</evaluation>
<feedback>
What needs improvement and why.
</feedback>
"""

generator_prompt = """
Your goal is to complete the task based on <user input>. If there are feedback
from your previous generations, you should reflect on them to improve your solution

Output your answer concisely in the following format:

<thoughts>
[Your understanding of the task and feedback and how you plan to improve]
</thoughts>

<response>
[Your code implementation here]
</response>
"""

task = """
<user input>
Implement a Stack with:
1. push(x)
2. pop()
3. getMin()
All operations should be O(1).
</user input>
"""

loop(task, evaluator_prompt, generator_prompt)


=== GENERATION START ===
Thoughts:

Implement a stack that supports push, pop, and getMin in O(1) time. The classic approach is to maintain an auxiliary min stack that tracks the minimum at each level.


Generated:

class MinStack:
    def __init__(self):
        self.stack = []
        self.min_stack = []  # stores current min at each push

    def push(self, x):
        self.stack.append(x)
        # Push the new minimum (either x or the current min)
        if not self.min_stack:
            self.min_stack.append(x)
        else:
            self.min_stack.append(min(x, self.min_stack[-1]))

    def pop(self):
        if not self.stack:
            raise IndexError("pop from empty stack")
        self.min_stack.pop()
        return self.stack.pop()

    def getMin(self):
        if not self.min_stack:
            raise IndexError("stack is empty")
        return self.min_stack[-1]


# Example usage
if __name__ == "__main__":
    s = MinStack()
    s.push(5)
    s.push(3)
    s.push

('\nclass MinStack:\n    def __init__(self):\n        self.stack = []\n        self.min_stack = []  # stores current min at each push\n\n    def push(self, x):\n        self.stack.append(x)\n        # Push the new minimum (either x or the current min)\n        if not self.min_stack:\n            self.min_stack.append(x)\n        else:\n            self.min_stack.append(min(x, self.min_stack[-1]))\n\n    def pop(self):\n        if not self.stack:\n            raise IndexError("pop from empty stack")\n        self.min_stack.pop()\n        return self.stack.pop()\n\n    def getMin(self):\n        if not self.min_stack:\n            raise IndexError("stack is empty")\n        return self.min_stack[-1]\n\n\n# Example usage\nif __name__ == "__main__":\n    s = MinStack()\n    s.push(5)\n    s.push(3)\n    s.push(7)\n    s.push(2)\n    s.push(4)\n\n    print("Min:", s.getMin())  # 2\n    s.pop()                    # removes 4\n    print("Min:", s.getMin())  # 2\n    s.pop()                   